# Module 11: What an embedding actually is

In your chat client and `semantic_search.py`, search "just worked" — a query found conversations that shared *no words* with it. This module opens the hood on the thing that makes that possible: the **embedding**.

An embedding is just a **list of numbers** (a *vector*) that captures the *meaning* of a piece of text. Two texts that mean similar things get vectors pointing in similar directions — and "how similar?" becomes a number you can actually compute.

> **Kernel note:** remember Module 10, where `import chromadb` errored? That was your Anaconda Python, which doesn't have it. This module needs the **memory-venv** kernel, where `chromadb` and the embedding model actually live. In the kernel picker (top-right), choose **Python (memory-venv)**. If you don't see it, tell Claude.

## The learning loop (7 steps)

1. **Read** the code
2. **Predict** what it does — *before* running
3. **Run** it and check
4. **Reflect** in your own words
5. **Ask** Claude if something's unclear
6. **Write** a small variant from a blank cell
7. **Fix** a deliberately broken version

# Lesson 1: an embedding is a list of numbers

**Read** the cell. **Predict:** how many numbers come back for a 4-word sentence? And will a *longer* sentence produce *more* numbers?

In [ ]:
from chromadb.utils import embedding_functions

embed = embedding_functions.DefaultEmbeddingFunction()  # the same local model your memory uses

vector = embed(["I love my cat"])[0]
print("how many numbers:", len(vector))
print("first 8 of them:", [round(n, 3) for n in vector[:8]])

**Reflect:** that list of numbers is the sentence's *meaning-fingerprint*. Two things to notice:
- it's **384 numbers long** no matter how short or long the sentence is, and
- the numbers mean nothing to a human on their own — what matters is how one vector *compares* to another. That's Lesson 2.

# Lesson 2: similarity = how aligned two vectors are

To compare two vectors we use **cosine similarity** — roughly, "do these two arrows point the same way?" It returns about **1.0** for same-meaning and near **0** for unrelated.

**Read** the helper below (you don't have to write it yet — just follow it). **Predict:** what should `cosine_similarity(v, v)` — a vector compared with *itself* — return?

In [ ]:
import math

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))      # multiply matching numbers, add them up
    norm_a = math.sqrt(sum(x * x for x in a))   # length of arrow a
    norm_b = math.sqrt(sum(x * x for x in b))   # length of arrow b
    return dot / (norm_a * norm_b)

# a vector compared with itself -- predict before running:
print(round(cosine_similarity(vector, vector), 3))

Now the real test. **Predict first:** which pair scores higher — the two animal sentences, or an animal sentence vs. the stock-market one?

In [ ]:
sentences = [
    "I love my cat",
    "my kitten is adorable",
    "the stock market fell sharply today",
]
v = embed(sentences)

print("cat   vs kitten:", round(cosine_similarity(v[0], v[1]), 3))
print("cat   vs stocks:", round(cosine_similarity(v[0], v[2]), 3))

**Reflect:** "cat" and "kitten" share barely any letters, yet their vectors are close — because the *model* learned they mean similar things. That's exactly why your semantic search surfaces a conversation about "telling me what I want to hear" when you search "sycophancy." Same mechanism — just 7,400 messages instead of 3.

## ✍️ Write (step 6)

From the blank cell, **author your own**:
- pick two sentences (any topic),
- embed them with `embed([...])`,
- print their `cosine_similarity`.

Predict the score first (high? low?), then run.

In [ ]:
# your code here


## 🔧 Fix (step 7)

The function below is **broken on purpose** — it returns a wrong number, not a similarity between -1 and 1. Read it, predict what's wrong, run to confirm, then fix it so `cat vs kitten` lands around 0.5–0.7 again. (Hint: compare it line by line with the working `cosine_similarity` above.)

In [ ]:
def broken_similarity(a, b):
    dot = sum(x + y for x, y in zip(a, b))     # is this really a dot product?
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b)

print(broken_similarity(v[0], v[1]))

## Where this shows up in your real code

You now understand the engine:
- **`embed_messages.py`** runs `embed(...)` over *every message in `memory.db`* and stores the vectors in `chroma_store`.
- **`semantic_search.py`** embeds your *query* the same way, then asks ChromaDB for the stored vectors with the highest cosine similarity to it.

**Module 12** will open those two real files and read them line by line — and now you'll recognize every move.